# [9.1] Refusal Directions and Safe Steering - Exercises

Build a safe refusal-direction validation ladder: construct a direction, score held-out activations, test steering, and reject shortcuts with random-direction and label-shuffle controls. The visible notebook uses toy tensors so you can inspect every number; the committed CUDA report checks the same contract on pinned Pythia/Qwen paths and the public refusal/compliance dataset.

<img src="../../instructions/assets/refusal_directions_safe_steering_validation_loop.svg" width="860">

The full section follows this loop: safe inputs -> activation caches -> candidate direction -> held-out separation -> steering/projection -> controls -> aggregate-only report. Failed controls send you back to the data or direction design.

In [ ]:
import json
import sys
from pathlib import Path

import torch as t

chapter = "chapter9_alignment_interpretability"
section = "part1_refusal_directions_safe_steering"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_refusal_directions_safe_steering.tests as tests
import part1_refusal_directions_safe_steering.utils as utils

from arena_ext.refusal_steering import (
    capability_degradation_report,
    direction_comparison_report,
    label_shuffle_control_report,
    mean_difference_direction,
    random_direction_control_report,
    refusal_direction_scores,
    refusal_separation_report,
    steering_effect_report,
)

GT_TIER = "GT-2"
EXERCISE_ID = "9_1_refusal_directions_and_safe_steering"
EXPECTED_RUNTIME = "seconds for toy contracts; minutes for the CUDA real-model preflights"
REQUIRES_GPU = True

MAIN = __name__ == "__main__"

## Candidate Refusal Directions

Compute a normalized mean-difference direction from refusal examples toward allowed examples.

> ```yaml
> Difficulty: easy
> Importance: high
> ```

<details>
<summary>Expected output</summary>

```text
All tests in `test_direction_smoke_test` passed!
```

</details>

<details>
<summary>Help - mean-difference direction</summary>

Average refusal activations, average non-refusal activations, subtract, and normalize. The sign convention matters: the direction should point toward refusal-labeled examples.

</details>

<details>
<summary>Common bugs</summary>

- Forgetting to normalize the direction.
- Flipping the sign by subtracting refusal from non-refusal.
- Treating this implementation check as real-model evidence.

</details>

<details>
<summary>Solution</summary>

```python
def direction_smoke_test() -> list[float]:
    refusal = t.tensor([[2.0, 0.0], [2.0, 1.0]])
    non_refusal = t.tensor([[0.0, 0.0], [0.0, 1.0]])
    return mean_difference_direction(refusal, non_refusal).tolist()
```

</details>

In [ ]:
def direction_smoke_test() -> list[float]:
    raise NotImplementedError()


tests.test_direction_smoke_test(direction_smoke_test)

## Projection Scores

Project activations onto the direction. This is the scalar evidence used by the separation, layer-sweep, and intervention reports.

> ```yaml
> Difficulty: easy
> Importance: high
> ```

<details>
<summary>Expected output</summary>

```text
scores: [2.0, 0.5]
All tests in `test_scores_smoke_test` passed!
```

</details>

<details>
<summary>Help - score sign</summary>

A higher score only means "more refusal-like" after you have fixed the direction sign and verified that refusal examples have higher mean score than non-refusal examples.

</details>

<details>
<summary>Common bugs</summary>

- Returning cosine similarities between examples instead of a projection onto the direction.
- Not normalizing the direction before scoring.

</details>

<details>
<summary>Solution</summary>

```python
def scores_smoke_test() -> list[float]:
    activations = t.tensor([[2.0, 0.0], [0.5, 1.0]])
    direction = t.tensor([1.0, 0.0])
    return refusal_direction_scores(activations, direction).tolist()
```

</details>

In [ ]:
def scores_smoke_test() -> list[float]:
    raise NotImplementedError()


tests.test_scores_smoke_test(scores_smoke_test)

## Held-Out Separation

Report whether the direction separates refusal-labeled examples from non-refusal examples using accuracy and signed margin.

> ```yaml
> Difficulty: medium
> Importance: high
> ```

<details>
<summary>Expected output</summary>

```text
accuracy: 1.0
margin: 2.25
All tests in `test_separation_smoke_test` passed!
```

</details>

<details>
<summary>Interpreting separation</summary>

The toy fixture should be perfect. In the real GT-2 path, the analogous held-out Qwen result is accuracy `1.0` and margin `6.576968312263489` on 32 held-out public dataset prompts.

</details>

<details>
<summary>Common bugs</summary>

- Reporting accuracy without signed margin.
- Evaluating on the same examples used to build the direction in the full experiment.
- Accepting a negative-margin separator.

</details>

<details>
<summary>Solution</summary>

```python
def separation_smoke_test() -> dict:
    activations = t.tensor([[2.0, 0.0], [3.0, 0.0], [0.0, 0.0], [0.5, 0.0]])
    labels = t.tensor([1, 1, 0, 0], dtype=t.bool)
    direction = t.tensor([1.0, 0.0])
    return refusal_separation_report(activations, labels, direction, min_accuracy=0.9).__dict__
```

</details>

In [ ]:
def separation_smoke_test() -> dict:
    raise NotImplementedError()


tests.test_separation_smoke_test(separation_smoke_test)

## Safe Steering Effects

Measure whether steering changes refusal rate on safe scores. In the full experiment, addition and projection-out are separate checks.

> ```yaml
> Difficulty: medium
> Importance: high
> ```

<details>
<summary>Expected output</summary>

```text
baseline_refusal_rate: 0.333...
steered_refusal_rate: 0.666...
All tests in `test_steering_smoke_test` passed!
```

</details>

<details>
<summary>Help - addition versus projection-out</summary>

Addition tests whether moving along the direction increases refusal evidence. Projection-out tests whether removing that direction decreases refusal evidence. A useful control knob should survive intervention, not just classification.

</details>

<details>
<summary>Common bugs</summary>

- Measuring raw score deltas instead of thresholded refusal-rate changes.
- Forgetting to specify the expected direction of change.
- Treating a steering effect as sufficient without capability and random controls.

</details>

<details>
<summary>Solution</summary>

```python
def steering_smoke_test() -> dict:
    baseline = t.tensor([0.2, 0.4, 0.7])
    steered = t.tensor([0.8, 0.9, 0.4])
    return steering_effect_report(baseline, steered, threshold=0.5, expected_direction="increase", min_rate_delta=0.3).__dict__
```

</details>

In [ ]:
def steering_smoke_test() -> dict:
    raise NotImplementedError()


tests.test_steering_smoke_test(steering_smoke_test)

## Capability Control

Bound ordinary benign-task degradation so the result is not just "the model got worse." 

> ```yaml
> Difficulty: easy
> Importance: high
> ```

<details>
<summary>Expected output</summary>

```text
degradation: 0.05
All tests in `test_capability_smoke_test` passed!
```

</details>

<details>
<summary>Interpreting capability controls</summary>

Capability controls are part of the safety boundary. The public GT-2 report stores aggregate behavioral accuracy `0.90625`, but it does not store raw completions.

</details>

<details>
<summary>Common bugs</summary>

- Only measuring refusal-rate improvement.
- Computing absolute score changes rather than baseline minus steered capability.
- Choosing the degradation bound after seeing the result.

</details>

<details>
<summary>Solution</summary>

```python
def capability_smoke_test() -> dict:
    baseline = t.tensor([0.9, 0.8])
    steered = t.tensor([0.85, 0.75])
    return capability_degradation_report(baseline, steered, max_degradation=0.1).__dict__
```

</details>

In [ ]:
def capability_smoke_test() -> dict:
    raise NotImplementedError()


tests.test_capability_smoke_test(capability_smoke_test)

## Random-Direction Control

Reject claims where a random direction produces a comparable effect.

> ```yaml
> Difficulty: easy
> Importance: high
> ```

<details>
<summary>Expected output</summary>

```text
margin: 0.35
All tests in `test_random_control_smoke_test` passed!
```

</details>

<details>
<summary>Help - random controls</summary>

High-dimensional spaces contain many directions that move scores. The target direction must beat matched random directions under the same metric and sign convention.

</details>

<details>
<summary>Common bugs</summary>

- Comparing the target direction only to zero.
- Using a random baseline without a fixed seed in the full experiment.
- Ignoring the sign of the intended effect.

</details>

<details>
<summary>Solution</summary>

```python
def random_control_smoke_test() -> dict:
    return random_direction_control_report(target_direction_delta=0.4, random_direction_delta=0.05, min_margin=0.2).__dict__
```

</details>

In [ ]:
def random_control_smoke_test() -> dict:
    raise NotImplementedError()


tests.test_random_control_smoke_test(random_control_smoke_test)

## Label-Shuffle Control

Reject directions that work about as well when labels are deliberately mismatched.

> ```yaml
> Difficulty: easy
> Importance: high
> ```

<details>
<summary>Expected output</summary>

```text
true_accuracy: 1.0
shuffled_accuracy: <= 0.5
All tests in `test_label_shuffle_smoke_test` passed!
```

</details>

<details>
<summary>Interpreting label shuffles</summary>

If shuffled labels recover the same direction, the apparent refusal feature may be a formatting, ordering, length, or category shortcut.

</details>

<details>
<summary>Common bugs</summary>

- Shuffling activations instead of labels and accidentally preserving pairings.
- Reporting only the shuffled accuracy without the true-vs-shuffled gap.

</details>

<details>
<summary>Solution</summary>

```python
def label_shuffle_smoke_test() -> dict:
    activations = t.tensor([[3.0, 0.0], [2.5, 0.0], [0.0, 0.0], [0.2, 0.0]])
    labels = t.tensor([1, 1, 0, 0], dtype=t.bool)
    return label_shuffle_control_report(activations, labels, min_accuracy_gap=0.25).__dict__
```

</details>

In [ ]:
def label_shuffle_smoke_test() -> dict:
    raise NotImplementedError()


tests.test_label_shuffle_smoke_test(label_shuffle_smoke_test)

## Candidate Comparison

Compare candidate direction-construction methods rather than reporting only a favorite direction.

> ```yaml
> Difficulty: easy
> Importance: medium
> ```

<details>
<summary>Expected output</summary>

```text
best_method: mean_difference
best_score: 0.95
All tests in `test_comparison_smoke_test` passed!
```

</details>

<details>
<summary>Help - fair method comparison</summary>

All candidate methods should be scored on the same held-out examples and with the same controls. Preserve method names so a reviewer can see what won.

</details>

<details>
<summary>Common bugs</summary>

- Sorting scores in ascending order.
- Dropping method names.
- Comparing train-set and held-out scores as if they were equivalent.

</details>

<details>
<summary>Solution</summary>

```python
def comparison_smoke_test() -> dict:
    return direction_comparison_report({"mean_difference": 0.95, "probe": 0.9, "sae_feature": 0.85, "gemma_scope": 0.8}).__dict__
```

</details>

In [ ]:
def comparison_smoke_test() -> dict:
    raise NotImplementedError()


tests.test_comparison_smoke_test(comparison_smoke_test)

## Notebook Contract

The CPU smoke contract aggregates the toy checks. The section's GT-2 acceptance comes from the committed CUDA real-model preflight report, not from these toy tensors alone.

<details>
<summary>Expected output</summary>

```text
All tests in `test_notebook_contract` passed!
```

</details>

<details>
<summary>Help - what this does and does not prove</summary>

The smoke contract proves that your implementation returns the right structured fields. It does not rerun Pythia/Qwen, and it does not prove the GT-2 public-dataset result. That evidence is checked by `run_gpu_test` below.

</details>

In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    raise NotImplementedError()


tests.test_notebook_contract(run_smoke_test)

## Signature Result

<img src="../../instructions/assets/refusal_directions_safe_steering_signature_result.svg" width="860">

The committed CUDA report records GT-2 held-out accuracy `1.0`, held-out margin `6.576968312263489`, best layer `23`, PC1 variance fraction `0.25265154242515564`, projection delta `-0.8125`, random margin gap `6.42253303527832`, and peak VRAM `1.9031662940979004 GB`.

<details>
<summary>Interpreting the result</summary>

The strongest evidence is the combination of held-out separation, projection-out behavior, and failed random/label-shuffle controls. The addition effect on the public dataset is small, so this is a scoped control result rather than proof that refusal is literally one-dimensional.

</details>

In [ ]:
def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    tests.test_committed_gpu_report_matches_refusal_direction_contract(report)
    gpu = report["metrics"]["gpu_test"]
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = run_gpu_test(max_vram_gb=24.0)
{key: gpu[key] for key in [
    "real_lm_category_heldout_accuracy",
    "instruction_refusal_allowed_add_delta",
    "instruction_refusal_projection_delta",
    "gt2_refusal_direction_heldout_accuracy",
    "gt2_refusal_direction_layer_sweep_best_layer",
    "gt2_refusal_direction_pc1_variance_fraction",
    "gt2_refusal_direction_projection_delta",
    "peak_vram_gb",
]}

## Limitations

- The toy cells teach implementation mechanics; they are not real-model evidence.
- The Pythia path is a safe hidden-state category proxy, not instruction refusal replication.
- The Qwen no-generation path checks logit-score interventions, not generated behavior.
- The public GT-2 path stores aggregate metrics and hashes only; it is not a broad deployment-safety evaluation.
- PCA/SVD evidence supports the control story but does not prove refusal is literally one-dimensional.